<div style="border:2px solid #b0c4b1; padding:20px; border-radius:6px;">
  <h3 style="color:#333; margin:0; font-size:35px;">Preprocessing</h3>
</div>

In [ ]:

data_dir = Path('data')
subjects = [f.name for f in data_dir.iterdir() if f.is_dir()]
print(subjects)

['sub-LTP063']


## Final Pipeline

nog toevoegen:
- filter op aantal trials
- performance?

In [1]:
# ---------------------------
# Import packages
# ---------------------------
import mne
import numpy as np
import pandas as pd
import json
import gc

from pathlib import Path
import warnings
from collections import Counter
from mne.preprocessing import ICA
from mne_icalabel import label_components
from functions import preprocess_events

# ---------------------------
# Settings
# ---------------------------
subjects = ['sub-LTP063']
sessions = ['ses-0', 'ses-1', 'ses-2', 'ses-3', 'ses-4', 'ses-5',
            'ses-6', 'ses-7', 'ses-8', 'ses-9', 'ses-10', 'ses-11',
            'ses-12', 'ses-13', 'ses-14', 'ses-15', 'ses-16', 'ses-17',
            'ses-18', 'ses-19']

output_dir = Path('./derivatives')
output_dir.mkdir(exist_ok=True)

face_channels = [
    'E1','E2','E3','E9','E10','E14','E15','E16','E17','E18',
    'E21','E22','E23','E26','E27','E32','E33','E116','E122',
    'E123','E125','E128','E8','E25','E126','E127'
]
mastoids = ['E57', 'E100']

# Kanalen die nooit als bad gemarkeerd mogen worden
protected_channels = face_channels + mastoids + ['Cz']

low_pass  = 30
baseline  = (-0.2, 0)
epoch_time = (-1.0, 2.2)

rt_min = 0.3
rt_max = 2.0

bad_ch_epoch_threshold = 0.10  # kanaal bad als het in >30% van epochs afgekeurd wordt
force_reprocess = False

# ---------------------------
# Main loop
# ---------------------------
for sub in subjects:
    for ses in sessions:

        print(f"\nProcessing {sub} | {ses}")

        out_file = output_dir / f'{sub}_{ses}_epo.fif'
        if out_file.exists() and not force_reprocess:
            print(f"Already processed, skipping: {out_file}")
            continue

        raw_path    = f'data/{sub}/{ses}/eeg/{sub}_{ses}_task-ltpFR_eeg.edf'
        events_path = f'data/{sub}/{ses}/eeg/{sub}_{ses}_task-ltpFR_events.tsv'
        json_path   = f'data/{sub}/{ses}/eeg/{sub}_{ses}_task-ltpFR_eeg.json'

        if not Path(raw_path).exists():
            print(f"Skipping {sub} | {ses}: file not found")
            continue

        # ---------------------------
        # Laden
        # ---------------------------
        raw = mne.io.read_raw_edf(raw_path, preload=False, verbose=False)

        if 'E129' in raw.ch_names:
            raw.rename_channels({'E129': 'Cz'})

        # ---------------------------
        # Montage
        # ---------------------------
        with open(json_path) as f:
            eeg_json = json.load(f)

        montage_name = eeg_json.get("CapManufacturersModelName", "")

        if "HydroCel" in montage_name:
            montage = mne.channels.make_standard_montage('GSN-HydroCel-129')
            print("Using HydroCel montage")
        else:
            print(f"Unknown montage: {montage_name}, skipping")
            continue

        raw.set_montage(montage, match_case=False, on_missing='ignore')

        # ---------------------------
        # Kanalen droppen
        # ---------------------------
        # Face channels
        raw.drop_channels([ch for ch in face_channels if ch in raw.ch_names])

        # Cz droppen (referentie-elektrode, SD = 0 na re-referencing)
        if 'Cz' in raw.ch_names:
            raw.drop_channels(['Cz'])

        # ---------------------------
        # Filteren + resamplen + referentie
        # ---------------------------
        raw.load_data()
        raw.notch_filter(freqs=50, picks='eeg', verbose=False)
        raw.filter(0.1, low_pass, verbose=False)   # 0.1 Hz high-pass verwijdert DC-drift
        raw.resample(256, npad="auto")
        raw.set_eeg_reference(ref_channels=mastoids, verbose=False)

        # ---------------------------
        # Events laden & filteren
        # ---------------------------
        events_raw = pd.read_csv(events_path, sep='\t')
        events     = preprocess_events(events_raw)

        # Taakprestatie check
        performance = events['correct'].mean()
        if performance < 0.55:
            print(f"WARNING: Low performance ({performance:.2%}), skipping session")
            continue
        print(f"Performance: {performance:.2%}")

        total_trials = len(events)
        print(f"Total trials: {total_trials}")

        events_rt = events[(events['RT'] > rt_min) & (events['RT'] < rt_max)]
        print(f"Removed by RT filter: {total_trials - len(events_rt)} ({total_trials - len(events_rt)}/{total_trials})")

        event_dict      = {'RECOG_TARGET': 1, 'RECOG_LURE': 2}
        events_filtered = events_rt[events_rt['trial_type'].isin(event_dict)]
        print(f"Trials after filtering: {len(events_filtered)}")

        if len(events_filtered) < 20:
            print(f"WARNING: Too few trials ({len(events_filtered)}), skipping session")
            continue

        events = events_filtered

        # MNE events array (sample-nummers NA resample)
        event_samples = (events['onset'] * raw.info['sfreq']).astype(int)
        event_ids     = events['trial_type'].map(event_dict).values

        events_mne = np.column_stack([
            event_samples,
            np.zeros(len(event_samples), int),
            event_ids
        ])

        # ---------------------------
        # Stap 1: Bad channel detectie op continue data
        # ---------------------------
        data_arr   = raw.get_data(picks='eeg')
        eeg_picks  = mne.pick_types(raw.info, eeg=True)
        eeg_ch_names = [raw.ch_names[i] for i in eeg_picks]

        # Criterium 1: drift (>5 SD van gemiddelde)
        ch_means = np.mean(data_arr, axis=1)
        bad_sd   = [
            eeg_ch_names[i] for i, m in enumerate(ch_means)
            if np.abs(m - np.mean(ch_means)) > 5 * np.std(ch_means)
        ]

        # Criterium 2: amplitude >500µV in meer dan 20% van de tijd
        bad_amp = [
            eeg_ch_names[i] for i, ch in enumerate(data_arr)
            if np.mean(np.abs(ch) > 500e-6) > 0.20
        ]

        # Criterium 3: variantie >3 SD boven gemiddelde
        ch_std    = np.std(data_arr, axis=1)
        bad_var   = [
            eeg_ch_names[i] for i, s in enumerate(ch_std)
            if s > np.mean(ch_std) + 3 * np.std(ch_std)
        ]

        # Combineer en bescherm mastoïden + Cz
        initial_bad = list(set(bad_sd + bad_amp + bad_var))
        initial_bad = [ch for ch in initial_bad if ch not in protected_channels]

        print(f"\nInitial bad channels:")
        print(f"  SD-based:        {bad_sd}")
        print(f"  Amplitude-based: {bad_amp}")
        print(f"  Variance-based:  {bad_var}")
        print(f"  Combined:        {initial_bad}")

        raw.info['bads'] = initial_bad
        if initial_bad:
            raw.interpolate_bads(reset_bads=True)
            print(f"  Interpolated:    {initial_bad}")

        # ---------------------------
        # ICA (op continue data, 1–100 Hz voor ICLabel)
        # ---------------------------
        raw_ica = raw.copy().filter(1.0, 100.0, verbose=False)

        mne.preprocessing.annotate_amplitude(
            raw_ica,
            peak=500e-6,
            flat=1e-6,
            picks='eeg',
            verbose=False
        )

        n_bad_seg = sum(1 for ann in raw_ica.annotations if ann['description'].startswith('BAD'))
        print(f"\nBad segments for ICA: {n_bad_seg}")

        ica = ICA(
            n_components=0.99,
            method='infomax',
            fit_params=dict(extended=True),
            random_state=42,
            max_iter=1024
        )

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ica.fit(raw_ica, verbose=False)

        ic_labels = label_components(raw_ica, ica, method='iclabel')

        # ICA overzicht
        print("\n" + "="*40)
        print("ICA COMPONENT OVERVIEW")
        print("="*40)

        for i, (lab, prob) in enumerate(zip(ic_labels['labels'], ic_labels['y_pred_proba'])):
            print(f"IC {i:02d}: {lab:10s} | prob = {prob:.2f}")

        label_counts = Counter(ic_labels['labels'])
        print("\nComponent summary:")
        for k, v in label_counts.items():
            print(f"  {k}: {v}")

        eye_idx = [
            i for i, (lab, prob) in enumerate(zip(ic_labels['labels'], ic_labels['y_pred_proba']))
            if 'eye' in lab.lower() and prob > 0.75
        ]

        print("\nComponents marked for removal:")
        for i in eye_idx:
            print(f"  IC {i}: {ic_labels['labels'][i]} ({ic_labels['y_pred_proba'][i]:.2f})")

        # Sanity checks
        print("\nSanity checks:")
        n_eye_total = len([i for i, lab in enumerate(ic_labels['labels']) if 'eye' in lab.lower()])
        if n_eye_total == 0:
            print("  WARNING: No eye components detected")
        if len(label_counts) == 1:
            print("  WARNING: All components have same label")
        low_conf = [p for p in ic_labels['y_pred_proba'] if p < 0.6]
        if len(low_conf) > len(ic_labels['y_pred_proba']) * 0.5:
            print("  WARNING: Many low-confidence classifications")

        print("="*40 + "\n")

        ica.exclude = eye_idx
        print(f"Removed ICs: {eye_idx}")

        del raw_ica
        gc.collect()

        # ---------------------------
        # Epochen
        # ---------------------------
        epochs = mne.Epochs(
            raw,
            events_mne,
            event_id={'target': 1, 'lure': 2},
            tmin=epoch_time[0],
            tmax=epoch_time[1],
            baseline=None,
            preload=True,
            metadata=events.reset_index(drop=True),
            verbose=False
        )

        print(f"Epochs before cleaning: {len(epochs)}")

        # ---------------------------
        # Stap 2: Bad channel detectie op epochs
        # ---------------------------
        epochs_tmp = epochs.copy()
        epochs_tmp.drop_bad(reject=dict(eeg=500e-6))

        all_bad_chs = []
        for entry in epochs_tmp.drop_log:
            all_bad_chs.extend(entry)

        ch_counts = Counter(all_bad_chs)
        print("\nMost rejected channels (500µV):")
        for ch, count in ch_counts.most_common(10):
            pct = count / len(epochs) * 100
            print(f"  {ch}: {count} epochs ({pct:.1f}%)")

        # Kanalen in >30% van epochs → interpoleren
        epoch_bad = [
            ch for ch, count in ch_counts.items()
            if count > len(epochs) * bad_ch_epoch_threshold
            and ch not in protected_channels
        ]

        if epoch_bad:
            print(f"\nExtra bad channels via epochs: {epoch_bad}")
            raw.info['bads'] = epoch_bad
            raw.interpolate_bads(reset_bads=True)
            print(f"Interpolated: {epoch_bad}")

            # Epochs opnieuw aanmaken na herinterpolatie
            epochs = mne.Epochs(
                raw,
                events_mne,
                event_id={'target': 1, 'lure': 2},
                tmin=epoch_time[0],
                tmax=epoch_time[1],
                baseline=None,
                preload=True,
                metadata=events.reset_index(drop=True),
                verbose=False
            )
            print(f"Epochs after re-interpolation: {len(epochs)}")

        del epochs_tmp
        gc.collect()

        # ---------------------------
        # Stap 3: Finale cleaning
        # ---------------------------
        epochs.drop_bad(reject=dict(eeg=500e-6))
        print(f"Epochs after 500µV reject: {len(epochs)}")

        ica.apply(epochs)

        epochs.apply_baseline(baseline)

        epochs.drop_bad(reject=dict(eeg=200e-6))
        print(f"Epochs after 200µV reject: {len(epochs)}")

        # Waarschuwing als te weinig trials overblijven
        if len(epochs) == 0:
            print(f"WARNING: No epochs remain, skipping session")
            del raw, epochs, ica, ic_labels
            gc.collect()
            continue

        if len(epochs) < 0.5 * len(events):
            print(f"WARNING: Less than 50% of trials remain ({len(epochs)}/{len(events)})")

        print(f"Final epochs: {len(epochs)}")

        # ---------------------------
        # Opslaan
        # ---------------------------
        epochs.save(out_file, overwrite=True)
        print(f"Saved: {out_file}")

        del raw, epochs, ica, ic_labels
        gc.collect()

print("\nDone.")


Processing sub-LTP063 | ses-0
Already processed, skipping: derivatives\sub-LTP063_ses-0_epo.fif

Processing sub-LTP063 | ses-1
Unknown montage: Geodisic Sensor Net 200 v2.1, skipping

Processing sub-LTP063 | ses-2
Already processed, skipping: derivatives\sub-LTP063_ses-2_epo.fif

Processing sub-LTP063 | ses-3
Already processed, skipping: derivatives\sub-LTP063_ses-3_epo.fif

Processing sub-LTP063 | ses-4
Already processed, skipping: derivatives\sub-LTP063_ses-4_epo.fif

Processing sub-LTP063 | ses-5
Unknown montage: Geodisic Sensor Net 200 v2.1, skipping

Processing sub-LTP063 | ses-6
Already processed, skipping: derivatives\sub-LTP063_ses-6_epo.fif

Processing sub-LTP063 | ses-7
Already processed, skipping: derivatives\sub-LTP063_ses-7_epo.fif

Processing sub-LTP063 | ses-8
Unknown montage: Geodisic Sensor Net 200 v2.1, skipping

Processing sub-LTP063 | ses-9
Already processed, skipping: derivatives\sub-LTP063_ses-9_epo.fif

Processing sub-LTP063 | ses-10
Already processed, skipping:

C:\Users\charl\AppData\Local\Temp\ipykernel_34716\4229279854.py:223: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw_ica, ica, method='iclabel')



ICA COMPONENT OVERVIEW
IC 00: brain      | prob = 1.00
IC 01: channel noise | prob = 0.57
IC 02: other      | prob = 0.85
IC 03: other      | prob = 0.95
IC 04: other      | prob = 0.92
IC 05: other      | prob = 0.89
IC 06: brain      | prob = 1.00
IC 07: brain      | prob = 1.00
IC 08: other      | prob = 0.59
IC 09: brain      | prob = 0.74
IC 10: brain      | prob = 0.78
IC 11: other      | prob = 0.81
IC 12: muscle artifact | prob = 0.39
IC 13: brain      | prob = 1.00
IC 14: other      | prob = 0.86
IC 15: brain      | prob = 0.97
IC 16: other      | prob = 0.77
IC 17: brain      | prob = 0.64
IC 18: other      | prob = 0.90
IC 19: other      | prob = 0.88
IC 20: other      | prob = 0.97
IC 21: other      | prob = 0.94
IC 22: other      | prob = 0.90
IC 23: channel noise | prob = 0.99
IC 24: other      | prob = 0.87
IC 25: brain      | prob = 0.44
IC 26: other      | prob = 0.69
IC 27: other      | prob = 0.95
IC 28: other      | prob = 0.99
IC 29: other      | prob = 0.84
IC 30

In [2]:
# ---------------------------
# Post-preprocessing overview
# ---------------------------
import mne
import numpy as np
import pandas as pd
from pathlib import Path
from functions import preprocess_events

# ---------------------------
# Settings
# ---------------------------
subjects = ['sub-LTP063']
sessions = ['ses-0', 'ses-1', 'ses-2', 'ses-3', 'ses-4', 'ses-5',
            'ses-6', 'ses-7', 'ses-8', 'ses-9', 'ses-10', 'ses-11',
            'ses-12', 'ses-13', 'ses-14', 'ses-15', 'ses-16', 'ses-17',
            'ses-18', 'ses-19']

output_dir = Path('./derivatives')

rt_min     = 0.3
rt_max     = 2.0
event_dict = {'RECOG_TARGET': 1, 'RECOG_LURE': 2}

# ---------------------------
# Overview per subject
# ---------------------------
for sub in subjects:

    print(f"\n{'='*65}")
    print(f"  OVERVIEW: {sub}")
    print(f"{'='*65}")
    print(f"  {'Session':<10} {'Status':<14} {'Total':>7} {'Target':>8} {'Lure':>7} {'% retained':>11}")
    print(f"  {'-'*10} {'-'*14} {'-'*7} {'-'*8} {'-'*7} {'-'*11}")

    rows = []

    for ses in sessions:
        epo_file    = output_dir / f'{sub}_{ses}_epo.fif'
        events_path = f'data/{sub}/{ses}/eeg/{sub}_{ses}_task-ltpFR_events.tsv'

        if not epo_file.exists():
            print(f"  {ses:<10} {'not processed':<14}")
            rows.append({'session': ses, 'processed': False,
                         'total': 0, 'target': 0, 'lure': 0, 'pct': np.nan})
            continue

        try:
            epochs = mne.read_epochs(epo_file, verbose=False, preload=False)

            n_total  = len(epochs)
            n_target = len(epochs['target']) if 'target' in epochs.event_id else 0
            n_lure   = len(epochs['lure'])   if 'lure'   in epochs.event_id else 0

            # Recalculate original trial count from events TSV
            if Path(events_path).exists():
                events_raw = pd.read_csv(events_path, sep='\t')
                events     = preprocess_events(events_raw)
                events_rt  = events[(events['RT'] > rt_min) & (events['RT'] < rt_max)]
                n_original = len(events_rt[events_rt['trial_type'].isin(event_dict)])
                pct        = n_total / n_original * 100 if n_original > 0 else np.nan
                pct_str    = f"{pct:.1f}%"
            else:
                n_original = np.nan
                pct        = np.nan
                pct_str    = "n/a"

            print(f"  {ses:<10} {'✓ processed':<14} {n_total:>7} {n_target:>8} {n_lure:>7} {pct_str:>11}")
            rows.append({'session': ses, 'processed': True,
                         'total': n_total, 'target': n_target,
                         'lure': n_lure, 'pct': pct, 'n_original': n_original})

        except Exception as e:
            print(f"  {ses:<10} {'⚠ READ ERROR':<14}  ({e})")
            rows.append({'session': ses, 'processed': False,
                         'total': 0, 'target': 0, 'lure': 0, 'pct': np.nan})

    # ---------------------------
    # Summary
    # ---------------------------
    df    = pd.DataFrame(rows)
    df_ok = df[df['processed']]

    print(f"\n{'='*65}")
    print(f"  SUMMARY")
    print(f"{'='*65}")
    print(f"  Sessions found:      {len(sessions)}")
    print(f"  Sessions processed:  {df['processed'].sum()}")
    print(f"  Sessions missing:    {(~df['processed']).sum()}")

    if len(df_ok) > 0:
        print(f"\n  Total across all sessions:")
        print(f"    Epochs:            {df_ok['total'].sum()}")
        print(f"    Target (old):      {df_ok['target'].sum()}")
        print(f"    Lure (new):        {df_ok['lure'].sum()}")

        print(f"\n  Average per session:")
        print(f"    Epochs:            {df_ok['total'].mean():.1f}  (SD = {df_ok['total'].std():.1f})")
        print(f"    Target:            {df_ok['target'].mean():.1f}  (SD = {df_ok['target'].std():.1f})")
        print(f"    Lure:              {df_ok['lure'].mean():.1f}  (SD = {df_ok['lure'].std():.1f})")

        pct_vals = df_ok['pct'].dropna()
        if len(pct_vals) > 0:
            print(f"\n  % trials retained after cleaning:")
            print(f"    Mean:              {pct_vals.mean():.1f}%")
            print(f"    Min:               {pct_vals.min():.1f}%  ({df_ok.loc[pct_vals.idxmin(), 'session']})")
            print(f"    Max:               {pct_vals.max():.1f}%  ({df_ok.loc[pct_vals.idxmax(), 'session']})")

        low = df_ok[df_ok['total'] < 50]
        if len(low) > 0:
            print(f"\n  ⚠ Sessions with <50 epochs:")
            for _, row in low.iterrows():
                print(f"    {row['session']}: {int(row['total'])} epochs")

        print(f"\n  Target/lure balance per session:")
        for _, row in df_ok.iterrows():
            if row['total'] > 0:
                ratio = row['target'] / row['total'] * 100
                flag  = "  ⚠ imbalance" if ratio < 35 or ratio > 65 else ""
                print(f"    {row['session']:<10}: {int(row['target'])} target / "
                      f"{int(row['lure'])} lure  ({ratio:.0f}% target){flag}")

print(f"\n{'='*65}\n")


  OVERVIEW: sub-LTP063
  Session    Status           Total   Target    Lure  % retained
  ---------- -------------- ------- -------- ------- -----------
Trials zonder response geskipt: 1
  ses-0      ✓ processed        165       61     104       53.7%
  ses-1      not processed 
  ses-2      ✓ processed         69       38      31       21.9%
  ses-3      ✓ processed         82       17      65       25.9%
  ses-4      ✓ processed        220       97     123       70.5%
  ses-5      not processed 
  ses-6      ✓ processed        139       88      51       44.6%
  ses-7      ✓ processed        172       96      76       62.1%
  ses-8      not processed 
  ses-9      ✓ processed        171      129      42       73.1%
  ses-10     ✓ processed        167      131      36       69.6%
Trials zonder response geskipt: 3
  ses-11     ✓ processed        202      127      75       87.4%
  ses-12     ✓ processed        214      106     108       89.9%
  ses-13     ✓ processed        194      147